In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('D1.csv', encoding='utf-8-sig')

# Parse dates properly
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

# Target variable: encode result as integer
df['target'] = df['FTR'].map({'H': 0, 'D': 1, 'A': 2})

C:\Users\USER\AppData\Local\Temp\ipykernel_33688\2202230085.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target'] = df['FTR'].map({'H': 0, 'D': 1, 'A': 2})


In [3]:
df

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA,target
0,D1,2025-08-22,19:30,Bayern Munich,RB Leipzig,6,0,H,3,0,...,1.88,1.98,1.93,1.99,1.93,1.90,1.86,2.07,1.92,0
1,D1,2025-08-23,14:30,Ein Frankfurt,Werder Bremen,4,1,H,2,0,...,2.03,2.02,1.91,1.91,2.03,1.83,1.93,1.91,2.06,0
2,D1,2025-08-23,14:30,Freiburg,Augsburg,1,3,A,0,3,...,1.93,1.97,1.95,1.97,1.93,1.90,1.86,2.03,1.96,2
3,D1,2025-08-23,14:30,Heidenheim,Wolfsburg,1,3,A,1,1,...,1.83,2.06,1.87,2.03,1.85,1.97,1.81,2.12,1.88,2
4,D1,2025-08-23,14:30,Leverkusen,Hoffenheim,1,2,A,1,1,...,1.88,1.97,1.95,1.98,2.02,1.88,1.86,2.00,1.97,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,D1,2026-05-16,14:30,Freiburg,RB Leipzig,4,1,H,2,1,...,1.98,NaN,NaN,1.88,1.98,1.84,1.90,1.95,2.03,0
302,D1,2026-05-16,14:30,Ein Frankfurt,Stuttgart,2,2,D,0,2,...,1.80,NaN,NaN,2.00,1.86,1.93,1.79,2.09,1.90,1
303,D1,2026-05-16,14:30,Bayern Munich,FC Koln,5,1,H,3,1,...,1.88,NaN,NaN,2.00,1.88,1.95,1.80,2.03,1.93,0
304,D1,2026-05-16,14:30,Heidenheim,Mainz,0,2,A,0,2,...,1.88,NaN,NaN,1.98,1.90,1.89,1.83,2.06,1.92,2


In [4]:
# Rolling form: last 5 games for each team
def compute_rolling_form(df, window=5):
    df = df.copy()
    team_stats = {}  # store running history per team
    
    home_form, away_form = [], []
    home_goals_scored, home_goals_conceded = [], []
    away_goals_scored, away_goals_conceded = [], []
    
    for _, row in df.iterrows():
        h, a = row['HomeTeam'], row['AwayTeam']
        
        # Get last N results for each team
        h_hist = team_stats.get(h, [])
        a_hist = team_stats.get(a, [])
        
        # Points from last 5 games (3=win, 1=draw, 0=loss)
        h_form = np.mean(h_hist[-window:]) if h_hist else 1.0
        a_form = np.mean(a_hist[-window:]) if a_hist else 1.0
        
        home_form.append(h_form)
        away_form.append(a_form)
        
        # Update history after recording (no lookahead)
        h_pts = 3 if row['FTR']=='H' else (1 if row['FTR']=='D' else 0)
        a_pts = 3 if row['FTR']=='A' else (1 if row['FTR']=='D' else 0)
        
        team_stats[h] = h_hist + [h_pts]
        team_stats[a] = a_hist + [a_pts]
    
    df['home_form_5'] = home_form
    df['away_form_5'] = away_form
    return df

df = compute_rolling_form(df)

In [5]:
df

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA,target,home_form_5,away_form_5
0,D1,2025-08-22,19:30,Bayern Munich,RB Leipzig,6,0,H,3,0,...,1.93,1.99,1.93,1.90,1.86,2.07,1.92,0,1.0,1.0
1,D1,2025-08-23,14:30,Ein Frankfurt,Werder Bremen,4,1,H,2,0,...,1.91,1.91,2.03,1.83,1.93,1.91,2.06,0,1.0,1.0
2,D1,2025-08-23,14:30,Freiburg,Augsburg,1,3,A,0,3,...,1.95,1.97,1.93,1.90,1.86,2.03,1.96,2,1.0,1.0
3,D1,2025-08-23,14:30,Heidenheim,Wolfsburg,1,3,A,1,1,...,1.87,2.03,1.85,1.97,1.81,2.12,1.88,2,1.0,1.0
4,D1,2025-08-23,14:30,Leverkusen,Hoffenheim,1,2,A,1,1,...,1.95,1.98,2.02,1.88,1.86,2.00,1.97,2,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,D1,2026-05-16,14:30,Freiburg,RB Leipzig,4,1,H,2,1,...,NaN,1.88,1.98,1.84,1.90,1.95,2.03,0,1.4,2.4
302,D1,2026-05-16,14:30,Ein Frankfurt,Stuttgart,2,2,D,0,2,...,NaN,2.00,1.86,1.93,1.79,2.09,1.90,1,0.8,1.6
303,D1,2026-05-16,14:30,Bayern Munich,FC Koln,5,1,H,3,1,...,NaN,2.00,1.88,1.95,1.80,2.03,1.93,0,2.6,1.0
304,D1,2026-05-16,14:30,Heidenheim,Mainz,0,2,A,0,2,...,NaN,1.98,1.90,1.89,1.83,2.06,1.92,2,2.0,0.8
